<!--
Licensed to the Apache Software Foundation (ASF) under one or more
contributor license agreements.  See the NOTICE file distributed with
this work for additional information regarding copyright ownership.
The ASF licenses this file to You under the Apache License, Version 2.0
(the "License"); you may not use this file except in compliance with
the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 04 · Experiment 4 — Basic + Wren + BI enrichment

On top of the onboarded base layer (experiment 3), upload the BI glossary, run **enrichment**, and activate the proposal. Enrichment overlays the glossary's semantics — aliases, synonyms (→ instructions), join relationships, and filtered metrics like Golden Yield — onto the introspected structure, and re-indexes so colloquial terms reach retrieval.

> Requires experiment 3 to have run (the project must already be onboarded). Enrichment returns **409** if there are no base models to enrich.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import eval_common as ec

# Edit here if your endpoints differ (Docker: agent_base_url=
# 'http://localhost:8090/ai-agent', superset_base_url='http://localhost:8090').
config = ec.EvalConfig.from_env()
client = ec.AgentClient(config)
me = client.login()
print('Authenticated as:', me.get('username') or me.get('email') or me)
print('DB id:', client.resolve_database_id(), '| schema:', config.schema_name)
questions = ec.parse_test_queries()
print('Loaded', len(questions), 'graded questions')

### Upload the glossary and enrich

In [ ]:
project = client.resolve_project()
glossary = ec.read_glossary()
doc = client.create_document_from_text(project['id'], glossary, 'bi_glossary.md')
print('Document:', doc['id'])
proposal = client.enrich(project['id'], doc['id'])
print('Proposed path:', proposal['proposed_path'])
print('Valid:', proposal['validation'].get('valid'))
print('Warnings:', proposal.get('warnings'))
print('--- proposed content (first 1500 chars) ---')
print(proposal['proposed_content'][:1500])

### Apply the proposal (supersedes the base files, triggers re-index)

In [ ]:
applied = client.apply_enrichment(project['id'], proposal)
active = [f['path'] for f in client.list_mdl_files(project['id'])
          if f.get('status') == 'active']
print('Active MDL files:', active)
# Sanity: materialize should report no duplicate-model warnings.
mat = client._ok(client._agent('POST',
    f"/agent/semantic-layer/projects/{project['id']}/materialize"), 'materialize')
print('Materialize file_count:', mat.get('file_count'), 'warnings:', mat.get('warnings'))

### Run the 15 questions against the enriched Wren layer

In [ ]:
results = ec.run_experiment(client, 'wren_bi', questions)
for r in results:
    print(f"{r['id']:>3} [{r['status']}] models={r['wren'].get('matched_models')} "
          f"{str(r['answer_summary'])[:70]}")

### Grade vs. ground truth

In [ ]:
graded = ec.grade_experiment(questions, results)
print('Score summary:', ec.score_summary(graded))
for g in graded:
    print(f"{g['id']:>3} {g['level']} {g['verdict']:<14} expected={g['expected']!r} got={str(g['got'])[:80]!r}")